# Embedding Word2Vec

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from TweetUtils import TweetUtils
from TweetModels import TweetModels

tweetUtils  = TweetUtils()
tweetModels = TweetModels()

CSVPREFIX = "W2V"
K = 5
KFOLD_EPOCHS = 200
VALIDATION_EPOCHS = 200
UNITS = [256, 512]#[64,128, 256, 512, 1024, 2048]

Using TensorFlow backend.


In [2]:
import keras
import tensorflow as tf
config = tf.ConfigProto()
config.gpu_options.allow_growth=True
sess = tf.Session(config=config)
keras.backend.set_session(sess)

In [3]:
all_posts, labeled_posts, number_of_tweets, tree_max_num_seq = tweetUtils.loadAllPosts()

Tweets etiquetados      :  818
no_in_data              :  65
number_of_tweets        :  753
all_posts               :  21741
number_of_retweets      :  297301
number_of_invalid_tweets:  6432
len(seqs_lens)   :  753
min__seq_len:  0
max__seq_len:  599
mean_seq_len:  36
mode_seq_len:  2


In [4]:
categories = ['true', 'false', 'unverified', 'non-rumor']
num_categories = len(categories)

#### Cambia según el embedding

Utilizando `generate_XY` debe definir `X, y, emb_size`

In [5]:
# build vocabulary and train model
import gensim
from gensim.utils import simple_preprocess
w2v300_emb_size = 300
WINDOW = 5
W2V_EPOCHS = 50
BATCH_SIZE = 128

documents = []
for k, v in labeled_posts.items():
    for t in v[1]:
        documents.append(simple_preprocess(all_posts[t[1]]['text']))
        
w2v300_model = gensim.models.word2vec.Word2Vec(
 documents,
 size=w2v300_emb_size,
 window=WINDOW,
 min_count=2,
 workers=1,
 iter=W2V_EPOCHS
)

#Train model
w2v300_model.train(documents, total_examples=len(documents), epochs=w2v300_model.epochs)
w2v300_model_vocab = w2v300_model.wv.vocab

W0728 13:09:39.827536 140132059211520 base_any2vec.py:596] Effective 'alpha' higher than previous training cycles


In [6]:
X, y, wonim = tweetUtils.generate_XY(all_posts, w2v300_model, w2v300_model_vocab, w2v300_emb_size, number_of_tweets, labeled_posts, tree_max_num_seq, categories)
emb_size = w2v300_emb_size

X.shape:  (753, 36, 300)
Y.shape:  (753, 4)
#Words not in model:  14031


# Redes Neuronales

In [7]:
#separación en datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15)

## K-Fold para obtener accuracy promedio

In [ ]:
history_lstm            = []
history_stacked_lstm    = []
history_gru             = []
history_stacked_gru     = []
history_bi_lstm         = []
history_bi_stacked_lstm = []
history_bi_gru          = []
history_bi_stacked_gru  = []
history_conv1D          = []
history_rcnn            = []

for unit_size in UNITS:
    print("####### K-FOLD FOR UNIT SIZE: ", unit_size)
    
    ##################### LSTM
    print("######### LSTM")
    model_lstm = tweetModels.create_model_LSTM(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    lstm_fold_results = tweetModels.perform_kfold(X_train, y_train, model_lstm, k = K, _epochs = KFOLD_EPOCHS)
    lstm_fold_results['unit_size'] = unit_size
    history_lstm.append(lstm_fold_results)
    
    ##################### Stacked-2 LSTM
    print("######### Stacked-2 LSTM")
    model_stacked_lstm = tweetModels.create_model_StackedLSTM(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    stacked_lstm_fold_results = tweetModels.perform_kfold(X_train, y_train, model_stacked_lstm, k = K, _epochs = KFOLD_EPOCHS)
    stacked_lstm_fold_results['unit_size'] = unit_size
    history_stacked_lstm.append(stacked_lstm_fold_results)
    
    ##################### GRU
    print("######### GRU")
    model_gru = tweetModels.create_model_GRU(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    gru_fold_results = tweetModels.perform_kfold(X_train, y_train, model_gru, k = K, _epochs = KFOLD_EPOCHS)
    gru_fold_results['unit_size'] = unit_size
    history_gru.append(gru_fold_results)
    
    ##################### Stacked-2 GRU
    print("######### Stacked-2 GRU")
    model_stacked_gru = tweetModels.create_model_StackedGRU(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    stacked_gru_fold_results = tweetModels.perform_kfold(X_train, y_train, model_stacked_gru, k = K, _epochs = KFOLD_EPOCHS)
    stacked_gru_fold_results['unit_size'] = unit_size
    history_stacked_gru.append(stacked_gru_fold_results)
    
    ##################### BI_LSTM
    print("######### BI_LSTM")
    model_bi_lstm = tweetModels.create_model_BI_LSTM(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    bi_lstm_fold_results = tweetModels.perform_kfold(X_train, y_train, model_bi_lstm, k = K, _epochs = KFOLD_EPOCHS)
    bi_lstm_fold_results['unit_size'] = unit_size
    history_bi_lstm.append(bi_lstm_fold_results)
    
    ##################### BI_Stacked_LSTM
    print("######### BI_StackedLSTM")
    model_bi_stacked_lstm = tweetModels.create_model_BI_StackedLSTM(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    bi_stacked_lstm_fold_results = tweetModels.perform_kfold(X_train, y_train, model_bi_stacked_lstm, k = K, _epochs = KFOLD_EPOCHS)
    bi_stacked_lstm_fold_results['unit_size'] = unit_size
    history_bi_stacked_lstm.append(bi_stacked_lstm_fold_results)

    ##################### BI_GRU
    print("######### BI_GRU")
    model_bi_gru = tweetModels.create_model_BI_GRU(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    bi_gru_fold_results = tweetModels.perform_kfold(X_train, y_train, model_bi_gru, k = K, _epochs = KFOLD_EPOCHS)
    bi_gru_fold_results['unit_size'] = unit_size
    history_bi_gru.append(bi_gru_fold_results)
    
    ##################### BI_Stacked_GRU
    print("######### BI_StackedGRU")
    model_bi_stacked_gru = tweetModels.create_model_BI_StackedGRU(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    bi_stacked_gru_fold_results = tweetModels.perform_kfold(X_train, y_train, model_bi_stacked_gru, k = K, _epochs = KFOLD_EPOCHS)
    bi_stacked_gru_fold_results['unit_size'] = unit_size
    history_bi_stacked_gru.append(bi_stacked_gru_fold_results)
    
    ##################### CONV1D
    print("######### Conv1D")
    model_conv1D = tweetModels.create_model_Conv1D(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    conv1D_fold_results = tweetModels.perform_kfold(X_train, y_train, model_conv1D, k = K, _epochs = KFOLD_EPOCHS)
    conv1D_fold_results['unit_size'] = unit_size
    history_conv1D.append(conv1D_fold_results)
    
    ##################### RCNN
    print("######### RCNN")
    model_rcnn = tweetModels.create_model_RCNN(tree_max_num_seq, emb_size, num_categories, _units = unit_size)
    rcnn_fold_results = tweetModels.perform_kfold(X_train, y_train, model_rcnn, k = K, _epochs = KFOLD_EPOCHS)
    rcnn_fold_results['unit_size'] = unit_size
    history_rcnn.append(rcnn_fold_results)
    
    
#### Almacenar resultados en CSV
## LSTM
history_lstm_df = pd.DataFrame(history_lstm)
history_lstm_df.to_csv(CSVPREFIX +'_history_lstm.csv', index = False)

## Stacked-2 LSTM
history_stacked_lstm_df = pd.DataFrame(history_stacked_lstm)
history_stacked_lstm_df.to_csv(CSVPREFIX + '_history_stacked_lstm.csv', index = False)

## GRU
history_gru_df = pd.DataFrame(history_gru)
history_gru_df.to_csv(CSVPREFIX + '_history_gru.csv', index = False)

## Stacked-2 GRU
history_stacked_gru_df = pd.DataFrame(history_stacked_gru)
history_stacked_gru_df.to_csv(CSVPREFIX + '_history_stacked_gru.csv', index = False)

##BI_LSTM
history_bi_lstm_df = pd.DataFrame(history_bi_lstm)
history_bi_lstm_df.to_csv(CSVPREFIX + '_history__bi_lstm.csv', index = False)


##BI_Stacked_LSTM
history_bi_stacked_lstm_df = pd.DataFrame(history_bi_stacked_lstm)
history_bi_stacked_lstm_df.to_csv(CSVPREFIX + '_history__bi_stacked_lstm.csv', index = False)


##BI_GRU
history_bi_gru_df = pd.DataFrame(history_bi_gru)
history_bi_gru_df.to_csv(CSVPREFIX + '_history__bi_gru.csv', index = False)


##BI_Stacked_GRU
history_bi_stacked_gru_df = pd.DataFrame(history_bi_stacked_gru)
history_bi_stacked_gru_df.to_csv(CSVPREFIX + '_history__bi_stacked_gru.csv', index = False)

##CONV1D
history_conv1D_df = pd.DataFrame(history_conv1D)
history_conv1D_df.to_csv(CSVPREFIX + '_history__conv1D.csv', index = False)

##RCNN
history_rcnn_df = pd.DataFrame(history_rcnn)
history_rcnn_df.to_csv(CSVPREFIX + '_history__rcnn.csv', index = False)

W0728 13:10:12.669842 140132059211520 deprecation_wrapper.py:119] From /home/srodriguez/miniconda/envs/jupyterhub/lib/python3.6/site-packages/keras/backend/tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0728 13:10:12.674361 140132059211520 deprecation_wrapper.py:119] From /home/srodriguez/miniconda/envs/jupyterhub/lib/python3.6/site-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0728 13:10:12.769810 140132059211520 deprecation_wrapper.py:119] From /home/srodriguez/miniconda/envs/jupyterhub/lib/python3.6/site-packages/keras/backend/tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.



####### K-FOLD FOR UNIT SIZE:  256
######### LSTM


W0728 13:10:13.239243 140132059211520 deprecation_wrapper.py:119] From /home/srodriguez/miniconda/envs/jupyterhub/lib/python3.6/site-packages/keras/backend/tensorflow_backend.py:133: The name tf.placeholder_with_default is deprecated. Please use tf.compat.v1.placeholder_with_default instead.

W0728 13:10:13.246733 140132059211520 deprecation.py:506] From /home/srodriguez/miniconda/envs/jupyterhub/lib/python3.6/site-packages/keras/backend/tensorflow_backend.py:3445: calling dropout (from tensorflow.python.ops.nn_ops) with keep_prob is deprecated and will be removed in a future version.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
W0728 13:10:13.266963 140132059211520 deprecation_wrapper.py:119] From /home/srodriguez/miniconda/envs/jupyterhub/lib/python3.6/site-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0728 13:10:13.281766 14013205921

Fold  1


## Utilizando datos de test en todo el modelo (datos que no se utilizaron en train)

In [ ]:
model_variants = [
    ("LSTM", tweetModels.create_model_LSTM),
    ("Stacked-2 LSTM", tweetModels.create_model_StackedLSTM),    
    ("GRU", tweetModels.create_model_GRU),
    ("Stacked-2 GRU", tweetModels.create_model_StackedGRU),
    ("BI_LSTM", tweetModels.create_model_BI_LSTM),
    ("BI_StackedLSTM", tweetModels.create_model_BI_StackedLSTM),
    ("BI_GRU", tweetModels.create_model_BI_GRU),
    ("BI_StackedGRU", tweetModels.create_model_BI_StackedGRU),
    ("Conv1D", tweetModels.create_model_Conv1D),
    ("RCNN", tweetModels.create_model_RCNN),
]

trained_models = {}
test_history = []

In [ ]:
for model_name, model_maker in model_variants:
    for unit_size in UNITS:
        
        print("######### Model: %s / Units: %d" % (model_name, unit_size))        
        
        current_model= model_maker(
            tree_max_num_seq,
            emb_size,
            num_categories,
            _units = unit_size
        )

        model_score, model_acc = tweetModels.perform_test( #entrena el modelo con datos de train, y acc para datos de test
            X_train, y_train,
            X_test, y_test,
            current_model,
            _epochs = VALIDATION_EPOCHS)
        
        model_index = "%s/%d" % (model_name, unit_size)
        trained_models[model_index] = current_model

        test_history.append({
            'model' : model_name,
            'score' : model_score,
            'acc'   : model_acc,
            'units' : unit_size
        })

In [ ]:
test_history_df = pd.DataFrame(test_history)
test_history_df.to_csv(CSVPREFIX + '_test_history.csv', index = False)

# Predecir

In [ ]:
def predictAll(_X, _y, _trained_models, tag = ""):
    y_predict_all = []
    column_names  = []
    for model_name, model in _trained_models.items():        
        print("\n###### Testing model %s" % (model_name))        
        units = int(model_name.split("/")[1])
        column_names.append(model_name)

        y_predict = tweetModels.predict_all(_X, _y, model, units)
        y_predict_all.append(y_predict.argmax(1))

    y_predict = pd.DataFrame(y_predict_all)
    y_predict['Model'] = column_names
    y_predict.to_csv(CSVPREFIX + '_predict%s.csv' % (tag), index = False)  
    return y_predict

## Solo test data 

In [ ]:
y_predict_df = predictAll(X_test, y_test, trained_models)

## Todos los datos

In [ ]:
y_predict_df_allData = predictAll(X, y, trained_models, tag="_allData")

### Guardamos los datos reales para plot de Matriz de Confusión en resultados

In [ ]:
pd.DataFrame(np.array([y_test.argmax(1)])).to_csv(CSVPREFIX + '_y_test.csv', index = False)
pd.DataFrame(np.array([y.argmax(1)])).to_csv(CSVPREFIX + '_y.csv', index = False)

#### Predecir utilizando todo el conjunto de entrenamiento

#### FIN

In [ ]:
print("Fin")